In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

# 1. 데이터 로드 (저장해둔 final_tmdb_data.csv)
base_path = '/content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘'
final_df = pd.read_csv(os.path.join(base_path, 'final_tmdb_data.csv'))

# 2. tmdbId 기준 중복 확인
# (tmdbId가 같다는 건 내용물(Soup)이 100% 똑같다는 뜻)
duplicates = final_df[final_df.duplicated(subset=['tmdbId'], keep=False)]

print(f"🔍 tmdbId가 완전히 동일한 중복 영화 수: {len(duplicates)}개")

if len(duplicates) > 0:
    print("\n[중복 데이터 샘플]")
    print(duplicates[['movieId', 'tmdbId', 'title']].sort_values(by='tmdbId').head(12))
    print("\n👉 논리: '이들은 감독판/국가별 버전 차이로 MovieLens ID가 분리된 것이며, 평점 데이터의 왜곡을 막기 위해 병합하지 않았습니다.'")
else:
    print("\n✅ 확인 결과: tmdbId가 겹치는 '진짜 중복'은 없습니다!")
    print("👉 논리: '제목만 같을 뿐, TMDB 기준으로는 모두 다른 영화(리메이크 등)이므로 삭제할 이유가 없습니다.'")

🔍 tmdbId가 완전히 동일한 중복 영화 수: 12개

[중복 데이터 샘플]
      movieId  tmdbId                                              title
4169     6003  4912.0             Confessions of a Dangerous Mind (2002)
4170     6003  4912.0             Confessions of a Dangerous Mind (2002)
9107   144606  4912.0             Confessions of a Dangerous Mind (2002)
9108   144606  4912.0             Confessions of a Dangerous Mind (2002)
624       791     NaN  Last Klezmer: Leopold Kozlowski, His Life and ...
843      1107     NaN                                       Loser (1991)
2141     2851     NaN                                    Saturn 3 (1980)
3027     4051     NaN  Horrors of Spider Island (Ein Toter Hing im Ne...
5533    26587     NaN                    Decalogue, The (Dekalog) (1989)
5855    32600     NaN                                        Eros (2004)
6060    40697     NaN                                          Babylon 5
7383    79299     NaN         No. 1 Ladies' Detective Agency, The (2008)

👉 논리: 

In [ ]:
import pandas as pd
import os

# 1. 경로 설정 및 데이터 로드
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘'

try:
    movies = pd.read_csv(os.path.join(BASE_PATH, 'movies.csv'))
    final_df = pd.read_csv(os.path.join(BASE_PATH, 'final_tmdb_data.csv'))
except FileNotFoundError:
    print("❌ 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

# 2. 매칭률 계산 로직

# (1) 분모: 전체 MovieLens 영화 개수
total_movies_count = len(movies)

# (2) 분자: TMDB 데이터(Soup)가 실제로 존재하는 영화 개수
# 병합 과정에서 TMDB 정보가 없으면 NaN이거나 빈 문자열('')로 되어 있을 것입니다.
final_df['metadata_soup'] = final_df['metadata_soup'].fillna('')
# 공백 제거 후 내용이 있는 것만 카운트
valid_tmdb_movies = final_df[final_df['metadata_soup'].str.strip() != '']
matched_count = len(valid_tmdb_movies)

# (3) 실패 개수
failed_count = total_movies_count - matched_count

# (4) 성공률 계산
success_rate = (matched_count / total_movies_count) * 100

# 3. 결과 출력
print(f"{'='*40}")
print(f"   🎬 [영화 개수 기준 매칭 분석]")
print(f"{'='*40}")
print(f"✅ 전체 영화 수 (MovieLens) : {total_movies_count:,} 개")
print(f"✅ 매칭 성공 수 (TMDB Soup 유) : {matched_count:,} 개")
print(f"❌ 매칭 실패 수 (TMDB Soup 무) : {failed_count:,} 개")
print(f"-"*40)
print(f"🎯 최종 매칭 성공률 : {success_rate:.2f}%")
print(f"{'='*40}")

# 4. (추가) 매칭 안 된 영화들은 도대체 뭐길래? (상위 5개 확인)
if failed_count > 0:
    print("\n🔍 [매칭 실패한 영화 샘플 (Top 5)]")
    # 실패한 영화 찾기 (전체 영화 목록에서 성공한 ID 제외)
    valid_ids = set(valid_tmdb_movies['movieId'])
    failed_df = movies[~movies['movieId'].isin(valid_ids)]

    print(failed_df.head(5))
    print("\n👉 원인: links.csv에 ID가 없거나, TMDB에서 삭제된 고전/비주류 영화일 가능성이 높습니다.")

   🎬 [영화 개수 기준 매칭 분석]
✅ 전체 영화 수 (MovieLens) : 9,742 개
✅ 매칭 성공 수 (TMDB Soup 유) : 9,744 개
❌ 매칭 실패 수 (TMDB Soup 무) : -2 개
----------------------------------------
🎯 최종 매칭 성공률 : 100.02%


In [ ]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr

# 1. 설정 및 데이터 로드
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘'

try:
    movies = pd.read_csv(os.path.join(BASE_PATH, 'movies.csv'))
    ratings = pd.read_csv(os.path.join(BASE_PATH, 'ratings.csv'))
    tags = pd.read_csv(os.path.join(BASE_PATH, 'tags.csv'))
    # TMDB 데이터 (결정적 판단 기준)
    final_df = pd.read_csv(os.path.join(BASE_PATH, 'final_tmdb_data.csv'))
except FileNotFoundError:
    print("❌ 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

# ---------------------------------------------------------
# [Step 1] 후보군 선정 (제목 정규화 후 중복 찾기)
# ---------------------------------------------------------
def normalize_title(title):
    title = re.sub(r'\s*\(\d{4}\)', '', str(title))
    if title.endswith(', The'): title = 'The ' + title[:-5]
    elif title.endswith(', A'): title = 'A ' + title[:-3]
    title = re.sub(r'[^a-zA-Z0-9 ]', '', title).lower().strip()
    return title

movies['norm_title'] = movies['title'].apply(normalize_title)
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)

# TMDB ID 매핑 (movieId 기준)
tmdb_map = final_df.set_index('movieId')['tmdbId'].to_dict()
movies['tmdbId'] = movies['movieId'].map(tmdb_map)

# 후보군 추출 (제목이 같은 영화 쌍 생성)
dup_candidates = movies[movies.duplicated(subset=['norm_title'], keep=False)].copy()
pairs = pd.merge(dup_candidates, dup_candidates, on='norm_title', suffixes=('_1', '_2'))
# 자기 자신 제외 & 중복 쌍 제거 (A-B만 남기고 B-A 제거)
pairs = pairs[pairs['movieId_1'] < pairs['movieId_2']].reset_index(drop=True)

print(f"🔍 검토 대상 영화 쌍(Pair): {len(pairs)}개")

# ---------------------------------------------------------
# [Step 2] 심화 분석 지표 계산 (EDA 코드 활용)
# ---------------------------------------------------------

# 1. 장르 유사도 (Jaccard)
def get_genre_sim(g1, g2):
    s1, s2 = set(g1.split('|')), set(g2.split('|'))
    return len(s1 & s2) / len(s1 | s2) if s1 or s2 else 0

pairs['genre_sim'] = pairs.apply(lambda x: get_genre_sim(x['genres_1'], x['genres_2']), axis=1)

# 2. 태그 유사도 (TF-IDF Cosine)
print("   - 태그 유사도 계산 중...")
tags['tag'] = tags['tag'].astype(str).fillna('')
movie_tags = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
tag_map = movie_tags.set_index('movieId')['tag']
tfidf = TfidfVectorizer(stop_words='english')

def calculate_tag_sim(row):
    t1, t2 = tag_map.get(row['movieId_1'], ""), tag_map.get(row['movieId_2'], "")
    if not t1 or not t2: return 0.0
    mat = tfidf.fit_transform([t1, t2])
    return cosine_similarity(mat[0:1], mat[1:2])[0][0]

pairs['tag_sim'] = pairs.apply(calculate_tag_sim, axis=1)

# 3. TMDB ID 일치 여부 (Key Signal)
# 둘 다 TMDB ID가 있고, 그 ID가 같으면 -> 빼박 중복
def check_tmdb_id(row):
    id1, id2 = row['tmdbId_1'], row['tmdbId_2']
    if pd.notna(id1) and pd.notna(id2):
        return 1.0 if id1 == id2 else 0.0 # 1이면 같은 영화, 0이면 다른 영화
    return -1.0 # 판단 불가 (둘 중 하나가 NaN)

pairs['tmdb_match'] = pairs.apply(check_tmdb_id, axis=1)

# ---------------------------------------------------------
# [Step 3] 최종 판단 로직 (Decision Maker)
# ---------------------------------------------------------
def decide_action(row):
    # 1순위: TMDB ID (가장 정확함)
    if row['tmdb_match'] == 1.0:
        return "DELETE (True Duplicate)" # ID가 같으면 무조건 삭제 대상
    elif row['tmdb_match'] == 0.0:
        return "KEEP (Different Movie)"  # ID가 다르면 무조건 유지 대상

    # 2순위: 내용 기반 추론 (TMDB ID가 없을 때)
    # 장르가 완전히 다르면(0.3 미만) -> 동명이작
    if row['genre_sim'] < 0.3:
        return "KEEP (Homonym)"

    # 장르가 같고 태그도 비슷하면 -> 중복 의심 (하지만 안전하게 유지 추천)
    if row['genre_sim'] >= 0.8 and row['tag_sim'] >= 0.5:
        return "REVIEW (Potential Duplicate)"

    return "KEEP (Remake/Series)"

pairs['decision'] = pairs.apply(decide_action, axis=1)

# ---------------------------------------------------------
# [Step 4] 결과 요약 및 출력 (수정됨)
# ---------------------------------------------------------
print(f"\n{'='*40}")
print(f"   📋 [중복 영화 처리 판단 결과]")
print(f"{'='*40}")

summary = pairs['decision'].value_counts()
print(summary)

print(f"\n🔍 [상세 샘플 확인]")

# 1. 삭제 대상 (진짜 중복) -> .str.startswith 사용으로 수정
dups = pairs[pairs['decision'].str.startswith("DELETE")]
if not dups.empty:
    print(f"\n🔴 [삭제 추천] 진짜 중복된 영화 ({len(dups)}개)")
    print(dups[['title_1', 'tmdbId_1', 'title_2', 'tmdbId_2']].head())
else:
    print("\n✅ 삭제할 '진짜 중복' 영화는 발견되지 않았습니다.")

# 2. 유지 대상 (리메이크/동명이작) -> .str.startswith 사용으로 수정
keeps = pairs[pairs['decision'].str.startswith("KEEP")]
if not keeps.empty:
    print(f"\n🟢 [유지 추천] 리메이크/동명이작 ({len(keeps)}개)")
    print(keeps[['title_1', 'year_1', 'title_2', 'year_2', 'genre_sim']].head())

# 3. 결론 멘트 생성
print(f"\n{'='*40}")
print(f"👨‍⚖️ [최종 판결]")
if len(dups) == 0:
    print("👉 \"분석 결과, 물리적으로 삭제해야 할 중복 데이터는 없습니다.\"")
    print("   \"제목이 같더라도 TMDB ID가 다르거나 장르가 다른 '별개의 영화'들입니다.\"")
    print("   \"따라서 중복 제거 프로세스 없이 그대로 진행하는 것이 데이터 손실을 막는 정당한 방법입니다.\"")
else:
    print(f"👉 \"분석 결과, {len(dups)}개의 행(진짜 중복)만 삭제하면 됩니다.\"")
    print("   \"나머지(대부분)는 리메이크/동명이작이므로 삭제하지 않고 보존해야 합니다.\"")

🔍 검토 대상 영화 쌍(Pair): 337개
   - 태그 유사도 계산 중...

   📋 [중복 영화 처리 판단 결과]
decision
KEEP (Different Movie)     333
KEEP (Remake/Series)         3
DELETE (True Duplicate)      1
Name: count, dtype: int64

🔍 [상세 샘플 확인]

🔴 [삭제 추천] 진짜 중복된 영화 (1개)
                                    title_1  tmdbId_1  \
238  Confessions of a Dangerous Mind (2002)    4912.0   

                                    title_2  tmdbId_2  
238  Confessions of a Dangerous Mind (2002)    4912.0  

🟢 [유지 추천] 리메이크/동명이작 (336개)
                  title_1  year_1                 title_2  year_2  genre_sim
0          Sabrina (1995)  1995.0          Sabrina (1954)  1954.0   1.000000
1       Persuasion (1995)  1995.0       Persuasion (2007)  2007.0   1.000000
2  Misérables, Les (1995)  1995.0  Misérables, Les (1998)  1998.0   0.500000
3  Misérables, Les (1995)  1995.0  Misérables, Les (2012)  2012.0   0.200000
4  Misérables, Les (1995)  1995.0  Misérables, Les (2000)  2000.0   0.333333

👨‍⚖️ [최종 판결]
👉 "분석 결과, 1개의 행(진짜 중복)만 삭제하면 됩니다.

In [5]:
import pandas as pd
import numpy as np
import re
import os

# =========================================================
# 1. 설정 및 데이터 로드 (에러 처리 강화)
# =========================================================
# 🚨 본인의 구글 드라이브 경로가 맞는지 다시 확인해주세요!
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘'

print("📂 데이터 로드 중...")

# 파일이 실제로 있는지 먼저 검사
movies_file = os.path.join(BASE_PATH, 'movies.csv')
tmdb_file = os.path.join(BASE_PATH, 'final_tmdb_data.csv')

if not os.path.exists(movies_file):
    # 경로가 틀렸으면 여기서 멈추고 에러 메시지를 띄움 (NameError 방지)
    raise FileNotFoundError(f"❌ 경로 오류! 파일을 찾을 수 없습니다.\n경로: {movies_file}")

try:
    movies = pd.read_csv(movies_file)
    final_df = pd.read_csv(tmdb_file)
    print("✅ 데이터 로드 완료!")
except Exception as e:
    raise ValueError(f"❌ 데이터 로드 중 알 수 없는 오류 발생: {e}")

# =========================================================
# 2. 전처리: 제목 정규화 & 연도 추출 & TMDB ID 매핑
# =========================================================
def normalize_title(title):
    title = re.sub(r'\s*\(\d{4}\)', '', str(title))
    if title.endswith(', The'): title = 'The ' + title[:-5]
    elif title.endswith(', A'): title = 'A ' + title[:-3]
    title = re.sub(r'[^a-zA-Z0-9 ]', '', title).lower().strip()
    return title

movies['norm_title'] = movies['title'].apply(normalize_title)
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)

# TMDB ID 매핑
tmdb_map = final_df.set_index('movieId')['tmdbId'].to_dict()
movies['tmdbId'] = movies['movieId'].map(tmdb_map)

# =========================================================
# 3. 후보군 선정 (제목이 같은 영화 쌍)
# =========================================================
dup_candidates = movies[movies.duplicated(subset=['norm_title'], keep=False)].copy()
pairs = pd.merge(dup_candidates, dup_candidates, on='norm_title', suffixes=('_1', '_2'))

# 자기 자신 제외 & 중복 쌍 제거 (A-B만 남기고 B-A 제거)
pairs = pairs[pairs['movieId_1'] < pairs['movieId_2']].reset_index(drop=True)

# =========================================================
# 4. 필터링: [개봉 연도 동일] AND [장르 유사도 >= 0.3]
# =========================================================

# 1. 개봉 연도 동일 필터링
same_year_pairs = pairs[pairs['year_1'] == pairs['year_2']].copy()

# 2. 장르 유사도 계산
def get_genre_sim(g1, g2):
    s1, s2 = set(g1.split('|')), set(g2.split('|'))
    return len(s1 & s2) / len(s1 | s2) if s1 or s2 else 0

same_year_pairs['genre_sim'] = same_year_pairs.apply(
    lambda x: get_genre_sim(x['genres_1'], x['genres_2']), axis=1
)

# 3. 장르 유사도 0.3 이상 필터링 (동명이작 제외)
high_risk_dups = same_year_pairs[same_year_pairs['genre_sim'] >= 0.3].copy()

# =========================================================
# 5. 최종 판결: 삭제 vs 유지 (TMDB ID 기준)
# =========================================================

def final_judgment(row):
    id1, id2 = row['tmdbId_1'], row['tmdbId_2']

    # Case 1: TMDB ID가 둘 다 있고, 서로 다름 -> "다른 버전(감독판 등)" -> 유지
    if pd.notna(id1) and pd.notna(id2) and id1 != id2:
        return "KEEP (Different Version)"

    # Case 2: TMDB ID가 같음 -> 빼박 중복 -> 삭제
    if pd.notna(id1) and pd.notna(id2) and id1 == id2:
        return "DELETE (Exact Match)"

    # Case 3: TMDB ID가 하나라도 없음(NaN) -> 정보 부족하지만 제목/연도/장르 같음 -> 중복 확률 매우 높음 -> 삭제
    return "DELETE (Hidden Duplicate)"

high_risk_dups['action'] = high_risk_dups.apply(final_judgment, axis=1)

# =========================================================
# 6. 결과 출력
# =========================================================
print(f"\n{'='*60}")
print(f"   🚨 [연도+장르 동일] 숨겨진 중복 탐지 결과")
print(f"{'='*60}")

summary = high_risk_dups['action'].value_counts()
print(summary)

# 삭제 대상 출력
delete_targets = high_risk_dups[high_risk_dups['action'].str.startswith("DELETE")]

if not delete_targets.empty:
    print(f"\n🗑️ [삭제 대상 리스트] 총 {len(delete_targets)}건")
    print(f"   -> 기준: 제목 동일 + 연도 동일 + 장르 유사도 0.3 이상 + (TMDB ID 같거나 없음)")
    display_cols = ['title_1', 'tmdbId_1', 'title_2', 'tmdbId_2', 'genre_sim', 'action']
    print(delete_targets[display_cols].head(10))

    # 삭제할 ID 목록
    ids_to_drop = delete_targets['movieId_2'].tolist()
    print(f"\n💡 [Action] 다음 movieId들을 제거하면 됩니다: {ids_to_drop}")
else:
    print("\n✅ 삭제할 숨겨진 중복 영화가 없습니다!")

# 유지 대상 출력
keep_targets = high_risk_dups[high_risk_dups['action'].str.startswith("KEEP")]
if not keep_targets.empty:
    print(f"\n🛡️ [유지 대상 (예외)] 총 {len(keep_targets)}건")
    print(keep_targets[['title_1', 'tmdbId_1', 'tmdbId_2']].head())

📂 데이터 로드 중...
✅ 데이터 로드 완료!

   🚨 [연도+장르 동일] 숨겨진 중복 탐지 결과
action
KEEP (Different Version)     2
DELETE (Hidden Duplicate)    2
DELETE (Exact Match)         1
Name: count, dtype: int64

🗑️ [삭제 대상 리스트] 총 3건
   -> 기준: 제목 동일 + 연도 동일 + 장르 유사도 0.3 이상 + (TMDB ID 같거나 없음)
                                    title_1  tmdbId_1  \
158                         Saturn 3 (1980)       NaN   
238  Confessions of a Dangerous Mind (2002)    4912.0   
296                             Eros (2004)       NaN   

                                    title_2  tmdbId_2  genre_sim  \
158                         Saturn 3 (1980)   19761.0   0.666667   
238  Confessions of a Dangerous Mind (2002)    4912.0   0.800000   
296                             Eros (2004)   39850.0   0.500000   

                        action  
158  DELETE (Hidden Duplicate)  
238       DELETE (Exact Match)  
296  DELETE (Hidden Duplicate)  

💡 [Action] 다음 movieId들을 제거하면 됩니다: [168358, 144606, 147002]

🛡️ [유지 대상 (예외)] 총 2건
                     

In [7]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. 설정 및 데이터 로드
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘'

print("📂 데이터 로드 및 전처리 중...")
try:
    movies = pd.read_csv(os.path.join(BASE_PATH, 'movies.csv'))
    tags = pd.read_csv(os.path.join(BASE_PATH, 'tags.csv'))
    # TMDB 데이터 (결정적 판단 기준)
    final_df = pd.read_csv(os.path.join(BASE_PATH, 'final_tmdb_data.csv'))
except FileNotFoundError:
    print("❌ 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

# ---------------------------------------------------------
# [Step 1] 후보군 선정 (제목 정규화 후 중복 찾기)
# ---------------------------------------------------------
def normalize_title(title):
    title = re.sub(r'\s*\(\d{4}\)', '', str(title))
    if title.endswith(', The'): title = 'The ' + title[:-5]
    elif title.endswith(', A'): title = 'A ' + title[:-3]
    title = re.sub(r'[^a-zA-Z0-9 ]', '', title).lower().strip()
    return title

movies['norm_title'] = movies['title'].apply(normalize_title)
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)

# TMDB ID 매핑
tmdb_map = final_df.set_index('movieId')['tmdbId'].to_dict()
movies['tmdbId'] = movies['movieId'].map(tmdb_map)

# 후보군 추출
dup_candidates = movies[movies.duplicated(subset=['norm_title'], keep=False)].copy()
pairs = pd.merge(dup_candidates, dup_candidates, on='norm_title', suffixes=('_1', '_2'))
# 자기 자신 제외 & 중복 쌍 제거
pairs = pairs[pairs['movieId_1'] < pairs['movieId_2']].reset_index(drop=True)

print(f"🔍 검토 대상 영화 쌍(Pair): {len(pairs)}개")

# ---------------------------------------------------------
# [Step 2] 심화 분석 지표 계산
# ---------------------------------------------------------
# 1. 장르 유사도
def get_genre_sim(g1, g2):
    s1, s2 = set(g1.split('|')), set(g2.split('|'))
    return len(s1 & s2) / len(s1 | s2) if s1 or s2 else 0

pairs['genre_sim'] = pairs.apply(lambda x: get_genre_sim(x['genres_1'], x['genres_2']), axis=1)

# 2. 태그 유사도
print("   - 태그 유사도 계산 중...")
tags['tag'] = tags['tag'].astype(str).fillna('')
movie_tags = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
tag_map = movie_tags.set_index('movieId')['tag']
tfidf = TfidfVectorizer(stop_words='english')

def calculate_tag_sim(row):
    t1, t2 = tag_map.get(row['movieId_1'], ""), tag_map.get(row['movieId_2'], "")
    if not t1 or not t2: return 0.0
    try:
        mat = tfidf.fit_transform([t1, t2])
        return cosine_similarity(mat[0:1], mat[1:2])[0][0]
    except:
        return 0.0

pairs['tag_sim'] = pairs.apply(calculate_tag_sim, axis=1)

# 3. TMDB ID 일치 여부
def check_tmdb_id(row):
    id1, id2 = row['tmdbId_1'], row['tmdbId_2']
    if pd.notna(id1) and pd.notna(id2):
        return 1.0 if id1 == id2 else 0.0
    return -1.0

pairs['tmdb_match'] = pairs.apply(check_tmdb_id, axis=1)

# ---------------------------------------------------------
# [Step 3] 최종 판단 로직 (Decision Maker) -> 여기가 에러 원인이었음
# ---------------------------------------------------------
def decide_action(row):
    # 1순위: TMDB ID
    if row['tmdb_match'] == 1.0:
        return "DELETE (True Duplicate)"
    elif row['tmdb_match'] == 0.0:
        return "KEEP (Different Movie)"

    # 2순위: 내용 기반 추론
    if row['genre_sim'] < 0.3:
        return "KEEP (Homonym)"

    # 3순위: 나머지 안전장치
    if row['genre_sim'] >= 0.8 and row['tag_sim'] >= 0.5:
        return "REVIEW (Potential Duplicate)"

    return "KEEP (Remake/Series)"

# 컬럼 생성 (이게 실행되어야 에러가 안 납니다!)
pairs['decision'] = pairs.apply(decide_action, axis=1)

# ---------------------------------------------------------
# [Step 4] 결과 요약 출력
# ---------------------------------------------------------
print(f"\n{'='*40}")
print(f"   📋 [중복 영화 처리 판단 결과]")
print(f"{'='*40}")
print(pairs['decision'].value_counts())

# ---------------------------------------------------------
# [Step 5] 리메이크/동명이작 데이터프레임 추출 (요청하신 부분)
# ---------------------------------------------------------
# 1. 'KEEP'으로 시작하는 결정만 필터링
keep_df = pairs[pairs['decision'].str.startswith("KEEP")].copy()

# 2. 보기 좋게 컬럼 정리
output_cols = [
    'decision',       # 판단 결과
    'title_1', 'year_1', 'tmdbId_1',
    'title_2', 'year_2', 'tmdbId_2',
    'genre_sim', 'tag_sim'
]

# 3. 데이터프레임 생성 및 확인
remake_homonym_list = keep_df[output_cols].sort_values(by=['decision', 'title_1'])

print(f"\n📂 [리메이크/동명이작 목록] 총 {len(remake_homonym_list)}건")
remake_homonym_list



📂 데이터 로드 및 전처리 중...
🔍 검토 대상 영화 쌍(Pair): 337개
   - 태그 유사도 계산 중...

   📋 [중복 영화 처리 판단 결과]
decision
KEEP (Different Movie)     333
KEEP (Remake/Series)         3
DELETE (True Duplicate)      1
Name: count, dtype: int64

📂 [리메이크/동명이작 목록] 총 336건


,decision,title_1,year_1,tmdbId_1,title_2,year_2,tmdbId_2,genre_sim,tag_sim
68,KEEP (Different Movie),12 Angry Men (1957),1957.0,389.0,12 Angry Men (1997),1997.0,12219.0,0.500000,0.0
334,KEEP (Different Movie),12 Chairs (1976),1976.0,27934.0,12 Chairs (1971),1971.0,31648.0,1.000000,0.0
56,KEEP (Different Movie),"20,000 Leagues Under the Sea (1954)",1954.0,173.0,"20,000 Leagues Under the Sea (1916)",1916.0,30266.0,0.500000,0.0
225,KEEP (Different Movie),3:10 to Yuma (1957),1957.0,14168.0,3:10 to Yuma (2007),2007.0,5176.0,0.500000,0.0
127,KEEP (Different Movie),About Last Night... (1986),1986.0,18169.0,About Last Night (2014),2014.0,222899.0,0.666667,0.0
...,...,...,...,...,...,...,...,...,...
234,KEEP (Different Movie),Zulu (1964),1964.0,14433.0,Zulu (2013),2013.0,185567.0,0.200000,0.0
313,KEEP (Different Movie),[REC] (2007),2007.0,8329.0,[REC]² (2009),2009.0,10664.0,0.666667,0.0
296,KEEP (Remake/Series),Eros (2004),2004.0,NaN,Eros (2004),2004.0,39850.0,0.500000,0.0
66,KEEP (Remake/Series),Loser (1991),1991.0,NaN,Loser (2000),2000.0,10642.0,0.500000,0.0


In [8]:
# 4. 파일 저장
save_path = os.path.join(BASE_PATH, 'remake_homonym_check_list.csv')
remake_homonym_list.to_csv(save_path, index=False)
print(f"\n💾 파일로 저장되었습니다: {save_path}")


💾 파일로 저장되었습니다: /content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘/remake_homonym_check_list.csv
